In [1]:
!pip install numpy==1.24.3

In [2]:
!pip install pandas numpy scikit-learn seaborn matplotlib

In [3]:
!pip install scikit-surprise

In [4]:
import pandas as pd

In [5]:
from google.colab import files
uploaded = files.upload()

Saving anime.csv to anime (1).csv


In [6]:
anime = pd.read_csv('anime.csv')
print(anime.head())

   MAL_ID                             Name Score  \
0       1                     Cowboy Bebop  8.78   
1       5  Cowboy Bebop: Tengoku no Tobira  8.39   
2       6                           Trigun  8.24   
3       7               Witch Hunter Robin  7.27   
4       8                   Bouken Ou Beet  6.98   

                                              Genres            English name  \
0    Action, Adventure, Comedy, Drama, Sci-Fi, Space            Cowboy Bebop   
1              Action, Drama, Mystery, Sci-Fi, Space  Cowboy Bebop:The Movie   
2  Action, Sci-Fi, Adventure, Comedy, Drama, Shounen                  Trigun   
3  Action, Mystery, Police, Supernatural, Drama, ...      Witch Hunter Robin   
4          Adventure, Fantasy, Shounen, Supernatural  Beet the Vandel Buster   

                      Japanese name   Type Episodes  \
0                         カウボーイビバップ     TV       26   
1                    カウボーイビバップ 天国の扉  Movie        1   
2                             トライガン     T

In [7]:
nRowsRead = 10
df1 = pd.read_csv('anime.csv', delimiter=',', nrows = nRowsRead)
df1.dataframeName = 'anime.csv'
nRow, nCol = df1.shape
print(f'There are {nRow} rows and {nCol} columns')

There are 10 rows and 35 columns


In [8]:
from google.colab import files
uploaded = files.upload()

Saving rating.csv.xlsx to rating.csv (1).xlsx


In [9]:
nRowsRead = 10
df2 = pd.read_excel('rating.csv.xlsx', nrows = nRowsRead)
df2.dataframeName = 'rating.csv.xlsx'
nRow, nCol = df2.shape
print(f'There are {nRow} rows and {nCol} columns')

There are 10 rows and 3 columns


In [10]:
ratings = pd.read_excel('rating.csv.xlsx')
print(ratings.head())

   user_id  anime_id  rating
0        1        20      -1
1        1        24      -1
2        1        79      -1
3        1       226      -1
4        1       241      -1


In [11]:
anime.dropna(inplace=True)

In [12]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel

anime['Genres'] = anime['Genres'].fillna('')

tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(anime['Genres'])

cosine_sim = linear_kernel(tfidf_matrix, tfidf_matrix)

anime_index = pd.Series(anime.index, index=anime['Name']).drop_duplicates()

def recommend_anime(title, top_n=10):
    idx = anime_index[title]
    sim_scores = list(enumerate(cosine_sim[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)[1:top_n+1]
    anime_indices = [i[0] for i in sim_scores]
    return anime['Name'].iloc[anime_indices]

recommend_anime('Naruto')

,Name
1574,Naruto: Shippuuden
12802,Boruto: Jump Festa 2016 Special
214,Rekka no Honoo
6319,Naruto: Honoo no Chuunin Shiken! Naruto vs. Ko...
7021,Naruto: Shippuuden Movie 6 - Road to Ninja
12492,Boruto: Naruto Next Generations
734,Dragon Ball Z
819,Dragon Ball Z Movie 11: Super Senshi Gekiha!! ...
892,Dragon Ball GT: Gokuu Gaiden! Yuuki no Akashi ...
4426,Dragon Ball Kai


In [13]:
import pandas as pd
from surprise import Dataset, Reader
from surprise.model_selection import train_test_split

ratings = pd.read_excel("rating.csv.xlsx")  # Make sure file name matches

reader = Reader(rating_scale=(1, 10))

data = Dataset.load_from_df(ratings[['user_id', 'anime_id', 'rating']], reader)

trainset, testset = train_test_split(data, test_size=0.2)

In [14]:
from surprise import Dataset, Reader, SVD
from surprise.model_selection import train_test_split
from surprise.accuracy import rmse

reader = Reader(rating_scale=(1, 10))
data = Dataset.load_from_df(ratings[['user_id', 'anime_id', 'rating']], reader)

trainset, testset = train_test_split(data, test_size=0.2)

model = SVD()
model.fit(trainset)
predictions = model.test(testset)

rmse(predictions)

RMSE: 2.2351


2.2350550792117665

In [15]:
print(anime.columns)

Index(['MAL_ID', 'Name', 'Score', 'Genres', 'English name', 'Japanese name',
       'Type', 'Episodes', 'Aired', 'Premiered', 'Producers', 'Licensors',
       'Studios', 'Source', 'Duration', 'Rating', 'Ranked', 'Popularity',
       'Members', 'Favorites', 'Watching', 'Completed', 'On-Hold', 'Dropped',
       'Plan to Watch', 'Score-10', 'Score-9', 'Score-8', 'Score-7', 'Score-6',
       'Score-5', 'Score-4', 'Score-3', 'Score-2', 'Score-1'],
      dtype='object')


In [16]:
def recommend_for_user(user_id, top_n=10):
    anime_ids = anime['MAL_ID'].tolist()
    predicted_ratings = [
        (anime_id, model.predict(user_id, anime_id).est) for anime_id in anime_ids
    ]
    top_anime = sorted(predicted_ratings, key=lambda x: x[1], reverse=True)[:top_n]
    top_ids = [x[0] for x in top_anime]
    return anime[anime['MAL_ID'].isin(top_ids)]['Name']

# Example
recommend_for_user(user_id=1)

,Name
1228,Nikutai Ten'i
2907,Daishikkin Helena
3394,Rensa Byoutou
4025,Houkago 2: Sayuri
5221,Highschool of the Dead
5709,HHH Triple Ecchi
6088,Mahou Shoujo Lyrical Nanoha: The Movie 2nd A's
6524,Soredemo Tsuma wo Aishiteru
6614,Sword Art Online
7266,High School DxD New


In [17]:
def recommend_for_user(user_id, top_n=10):
    anime_ids = anime['MAL_ID'].tolist()
    predicted_ratings = [
        (anime_id, model.predict(user_id, anime_id).est) for anime_id in anime_ids
    ]
    top_anime = sorted(predicted_ratings, key=lambda x: x[1], reverse=True)[:top_n]
    top_ids = [x[0] for x in top_anime]
    return anime[anime['MAL_ID'].isin(top_ids)]['Name']

# Example
recommend_for_user(user_id=2)

,Name
38,Beck
428,Mushishi
708,Hellsing Ultimate
2656,Code Geass: Hangyaku no Lelouch R2
2708,Ookami to Koushinryou
3230,Detroit Metal City
6620,Kuroko no Basket
9011,Nanatsu no Taizai
9895,Haikyuu!! Second Season
9913,Gintama°


In [18]:
print(ratings.dtypes)
print(ratings.head())

user_id     int64
anime_id    int64
rating      int64
dtype: object
   user_id  anime_id  rating
0        1        20      -1
1        1        24      -1
2        1        79      -1
3        1       226      -1
4        1       241      -1
